In [2]:
import numpy as np
import pandas as pd
from IPython.display import display, clear_output
import time # Import time module for pauses

# Define environment parameters
num_states = 3
num_actions = 2

# Define state and action labels for better readability
state_labels = ['S1', 'S2', 'S3 (Terminal)']
action_labels = ['R (Right)', 'L (Left)']

# Define the learning parameters
gamma = 0.9  # Discount factor
alpha = 0.5  # Learning rate
num_iterations = 100 # Number of iterations (epochs) to run the Bellman updates

# Initialize Q-table with zeros
# Rows correspond to states (S1, S2, S3)
# Columns correspond to actions (R, L)
Q = np.zeros((num_states, num_actions))

# Define the environment transitions and rewards:
# (current_state, action) -> {'next_state': next_state_index, 'reward': reward_value}
environment = {
    (0, 0): {'next_state': 1, 'reward': -1},  # S1 --R--> S2 (reward -1)
    (0, 1): {'next_state': 0, 'reward': -1},  # S1 --L--> S1 (reward -1)
    (1, 0): {'next_state': 2, 'reward': 10}, # S2 --R--> S3 (reward +10, S3 is terminal)
    (1, 1): {'next_state': 0, 'reward': -1},  # S2 --L--> S1 (reward -1)
    # S3 is a terminal state. No actions are taken *from* S3 to update its Q-values as a starting state.
    # The future reward from S3 is 0.
}

terminal_state_index = 2 # S3 is the terminal state

print("初始Q-table (所有Q值均为0):")
display(pd.DataFrame(Q, index=state_labels, columns=action_labels))

print("\n开始模拟贝尔曼方程的更新过程 (每秒更新一次Q-table):\n")

# Simulate the Q-learning process over several iterations
for iteration in range(num_iterations):
    # For each iteration, we update Q-values for all state-action pairs.
    # This is similar to value iteration but using the Q-learning update rule.
    for s in range(num_states):
        # We don't update Q-values for actions taken from a terminal state (S3).
        # Only states S1 and S2 can initiate actions.
        if s == terminal_state_index:
            continue # Skip terminal state as a starting state

        for a in range(num_actions):
            # Get next state and reward from the environment definition
            env_info = environment.get((s, a))
            if env_info is None:
                continue # Should not happen with a well-defined environment

            next_s = env_info['next_state']
            reward = env_info['reward']

            # Calculate the maximum Q-value for the next state (max_a' Q(s', a'))
            # If the next state is terminal, its future reward is 0.
            if next_s == terminal_state_index:
                max_next_q = 0.0
            else:
                # Get the maximum Q-value for all possible actions from the next state
                max_next_q = np.max(Q[next_s, :])

            # Bellman Equation update (Q-learning update rule):
            # Q(s, a) = Q(s, a) + alpha * [R(s, a) + gamma * max_a' Q(s', a') - Q(s, a)]
            old_q_value = Q[s, a]
            Q[s, a] = old_q_value + alpha * (reward + gamma * max_next_q - old_q_value)

    # Display the Q-table every iteration for dynamic update
    clear_output(wait=True) # Clear previous output
    print(f"第 {iteration + 1} 次迭代后的Q-table:")
    display(pd.DataFrame(Q, index=state_labels, columns=action_labels).round(4)) # Round for cleaner display
    time.sleep(1) # Pause for 1 second

print("\n所有迭代完成后，最终的Q-table:")
display(pd.DataFrame(Q, index=state_labels, columns=action_labels).round(4))


第 100 次迭代后的Q-table:


,R (Right),L (Left)
S1,8.0,6.2
S2,10.0,6.2
S3 (Terminal),0.0,0.0



所有迭代完成后，最终的Q-table:


,R (Right),L (Left)
S1,8.0,6.2
S2,10.0,6.2
S3 (Terminal),0.0,0.0
